# 📡 TelecomX LATAM — Análisis de Evasión de Clientes (Churn)

**Empresa:** Telecom X  
**Proyecto:** Churn de Clientes  
**Objetivo:** Identificar los factores que llevan a la pérdida de clientes para que el equipo de Data Science pueda desarrollar modelos predictivos y estrategias de retención.

---

# 📌 Extracción
Carga de datos desde la API de Telecom X en formato JSON y conversión a DataFrame de Pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Cargar datos desde la API (JSON) ──────────────────────────────────────────
url_api = 'https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json'
df_raw = pd.read_json(url_api)

print(f'✅ Datos cargados: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas')
df_raw.head(3)

In [ ]:
# El JSON tiene columnas anidadas (customer, phone, internet, account)
# Las normalizamos con json_normalize
import json, urllib.request

with urllib.request.urlopen(url_api) as response:
    data_json = json.load(response)

df = pd.json_normalize(data_json)
print(f'✅ DataFrame normalizado: {df.shape[0]} filas x {df.shape[1]} columnas')
print('\nColumnas:', df.columns.tolist())
df.head(3)

# 🔧 Transformación
Limpieza, tratamiento de nulos, corrección de tipos, creación de nuevas variables y estandarización.

## 2.1 Exploración inicial y estructura de datos

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print('\n=== Valores nulos ===')
print(df.isnull().sum())
print(f'\n=== Duplicados: {df.duplicated().sum()} ===')

In [ ]:
print('=== Info general ===')
df.info()

## 2.2 Limpieza y corrección de datos

In [ ]:
# ── Renombrar columnas para mayor claridad ────────────────────────────────────
df.rename(columns={
    'customerID'               : 'ID_Cliente',
    'Churn'                    : 'Evasion',
    'customer.gender'          : 'Genero',
    'customer.SeniorCitizen'   : 'Adulto_Mayor',
    'customer.Partner'         : 'Pareja',
    'customer.Dependents'      : 'Dependientes',
    'customer.tenure'          : 'Meses_Contrato',
    'phone.PhoneService'       : 'Servicio_Telefono',
    'phone.MultipleLines'      : 'Multiples_Lineas',
    'internet.InternetService' : 'Servicio_Internet',
    'internet.OnlineSecurity'  : 'Seguridad_Online',
    'internet.OnlineBackup'    : 'Backup_Online',
    'internet.DeviceProtection': 'Proteccion_Dispositivo',
    'internet.TechSupport'     : 'Soporte_Tecnico',
    'internet.StreamingTV'     : 'Streaming_TV',
    'internet.StreamingMovies' : 'Streaming_Peliculas',
    'account.Contract'         : 'Tipo_Contrato',
    'account.PaperlessBilling' : 'Factura_Electronica',
    'account.PaymentMethod'    : 'Metodo_Pago',
    'account.Charges.Monthly'  : 'Cargo_Mensual',
    'account.Charges.Total'    : 'Cargo_Total'
}, inplace=True)

print('✅ Columnas renombradas')
df.columns.tolist()

In [ ]:
# ── Corregir tipos de datos ───────────────────────────────────────────────────
# Cargo_Total viene como string (con espacios vacíos), convertir a numérico
df['Cargo_Total'] = pd.to_numeric(df['Cargo_Total'], errors='coerce')

# Evasion: convertir 'Yes'/'No' → 1/0
df['Evasion'] = df['Evasion'].map({'Yes': 1, 'No': 0})

print('Tipo de Evasion:', df['Evasion'].dtype)
print('Tipo de Cargo_Total:', df['Cargo_Total'].dtype)
print('Valores únicos Evasion:', df['Evasion'].unique())

In [ ]:
# ── Valores nulos después de la conversión ────────────────────────────────────
nulos = df.isnull().sum()
print('Columnas con nulos:')
print(nulos[nulos > 0])

# Rellenar Cargo_Total nulo con la mediana
mediana_total = df['Cargo_Total'].median()
df['Cargo_Total'].fillna(mediana_total, inplace=True)
print(f'\n✅ Nulos en Cargo_Total rellenados con mediana: ${mediana_total:.2f}')
print('Nulos restantes:', df.isnull().sum().sum())

In [ ]:
# ── Eliminar duplicados ───────────────────────────────────────────────────────
antes = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicados eliminados: {antes - len(df)}')
print(f'Registros finales: {len(df)}')

In [ ]:
# ── Estandarizar columnas binarias Yes/No → 1/0 ───────────────────────────────
cols_binarias = [
    'Pareja', 'Dependientes', 'Servicio_Telefono',
    'Factura_Electronica'
]
for col in cols_binarias:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Adulto_Mayor ya es 0/1
print('✅ Variables binarias estandarizadas')
df[cols_binarias + ['Adulto_Mayor']].head(3)

In [ ]:
# ── Crear columna Cuentas_Diarias ─────────────────────────────────────────────
df['Cuentas_Diarias'] = round(df['Cargo_Mensual'] / 30, 4)

print('✅ Columna Cuentas_Diarias creada')
print(df[['ID_Cliente', 'Cargo_Mensual', 'Cuentas_Diarias']].head())

In [ ]:
# ── Crear columna: Cantidad de servicios contratados ──────────────────────────
servicios = [
    'Servicio_Telefono', 'Multiples_Lineas', 'Seguridad_Online',
    'Backup_Online', 'Proteccion_Dispositivo', 'Soporte_Tecnico',
    'Streaming_TV', 'Streaming_Peliculas'
]
# Mapear Yes→1, No/No phone service/No internet service → 0
for col in servicios:
    df[col + '_bin'] = df[col].apply(lambda x: 1 if x == 'Yes' else 0)

servicios_bin = [c + '_bin' for c in servicios]
df['Num_Servicios'] = df[servicios_bin].sum(axis=1)

print('✅ Columna Num_Servicios creada')
print(df['Num_Servicios'].value_counts().sort_index())

In [ ]:
# ── Vista final del DataFrame limpio ─────────────────────────────────────────
print(f'Dataset final: {df.shape[0]} filas x {df.shape[1]} columnas')
df[['ID_Cliente','Evasion','Genero','Adulto_Mayor','Meses_Contrato',
    'Cargo_Mensual','Cargo_Total','Cuentas_Diarias','Num_Servicios']].head()

# 📊 Carga y Análisis Exploratorio de Datos (EDA)

## 3.1 Estadísticas Descriptivas

In [ ]:
cols_numericas = ['Meses_Contrato', 'Cargo_Mensual', 'Cargo_Total', 'Cuentas_Diarias', 'Num_Servicios']
df[cols_numericas].describe().round(2)

## 3.2 Distribución de la Variable Objetivo: Churn (Evasión)

In [ ]:
conteo_churn = df['Evasion'].value_counts()
pct_churn    = df['Evasion'].value_counts(normalize=True) * 100

print('=== Distribución de Churn ===')
print(f'Sin evasión (0): {conteo_churn[0]:,} clientes ({pct_churn[0]:.1f}%)')
print(f'Con evasión (1): {conteo_churn[1]:,} clientes ({pct_churn[1]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Torta
axes[0].pie(conteo_churn, labels=['No Evadió', 'Evadió'],
            autopct='%1.1f%%', colors=['#42A5F5', '#EF5350'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
            textprops={'fontsize': 12})
axes[0].set_title('Proporción de Churn', fontsize=14, fontweight='bold')

# Barras
bars = axes[1].bar(['No Evadió', 'Evadió'], conteo_churn,
                   color=['#42A5F5', '#EF5350'], edgecolor='white', width=0.5)
for bar, pct in zip(bars, pct_churn):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{bar.get_height():,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Cantidad de Clientes por Churn', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Cantidad de Clientes')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_ylim(0, conteo_churn.max() * 1.2)

plt.suptitle('Distribución de Evasión de Clientes (Churn)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3.3 Churn por Variables Categóricas

In [ ]:
def plot_churn_por_categoria(col, titulo, ax):
    tabla = df.groupby(col)['Evasion'].mean() * 100
    colores = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(tabla)))
    bars = ax.bar(tabla.index, tabla.values, color=colores, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Tasa de Churn (%)')
    ax.set_ylim(0, tabla.max() * 1.25)
    ax.tick_params(axis='x', rotation=15)
    ax.spines[['top', 'right']].set_visible(False)
    ax.axhline(y=df['Evasion'].mean()*100, color='gray', linestyle='--', alpha=0.6, label='Promedio global')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

plot_churn_por_categoria('Genero',          'Churn por Género',             axes[0,0])
plot_churn_por_categoria('Adulto_Mayor',    'Churn: Adulto Mayor',          axes[0,1])
plot_churn_por_categoria('Tipo_Contrato',   'Churn por Tipo de Contrato',   axes[0,2])
plot_churn_por_categoria('Metodo_Pago',     'Churn por Método de Pago',     axes[1,0])
plot_churn_por_categoria('Servicio_Internet','Churn por Servicio Internet', axes[1,1])
plot_churn_por_categoria('Factura_Electronica','Churn: Factura Electrónica',axes[1,2])

plt.suptitle('Tasa de Churn por Variables Categóricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3.4 Churn por Variables Numéricas (Distribuciones)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cols_num = ['Meses_Contrato', 'Cargo_Mensual', 'Cargo_Total']
titulos  = ['Meses de Contrato', 'Cargo Mensual (USD)', 'Cargo Total (USD)']
colores  = {'No Evadió': '#42A5F5', 'Evadió': '#EF5350'}

for ax, col, titulo in zip(axes, cols_num, titulos):
    for label, val, color in [('No Evadió', 0, '#42A5F5'), ('Evadió', 1, '#EF5350')]:
        subset = df[df['Evasion'] == val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_title(f'Distribución: {titulo}', fontsize=12, fontweight='bold')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Distribución de Variables Numéricas según Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por Churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, titulo in zip(axes, cols_num, titulos):
    data_no  = df[df['Evasion'] == 0][col]
    data_yes = df[df['Evasion'] == 1][col]
    bp = ax.boxplot([data_no, data_yes], patch_artist=True,
                    labels=['No Evadió', 'Evadió'],
                    medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('#42A5F5')
    bp['boxes'][1].set_facecolor('#EF5350')
    ax.set_title(f'Boxplot: {titulo}', fontsize=12, fontweight='bold')
    ax.set_ylabel(titulo)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Comparación de Variables Numéricas (Churn vs No Churn)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Churn por número de servicios contratados
churn_servicios = df.groupby('Num_Servicios')['Evasion'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(churn_servicios.index, churn_servicios.values, marker='o', markersize=9,
        linewidth=2.5, color='#AB47BC', markerfacecolor='white', markeredgewidth=2.5)
for x, y in zip(churn_servicios.index, churn_servicios.values):
    ax.annotate(f'{y:.1f}%', (x, y), xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold')
ax.fill_between(churn_servicios.index, churn_servicios.values, alpha=0.1, color='#AB47BC')
ax.set_title('Tasa de Churn según Cantidad de Servicios Contratados', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Servicios')
ax.set_ylabel('Tasa de Churn (%)')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 3.5 EXTRA — Matriz de Correlación

In [ ]:
# Seleccionar variables numéricas/binarias para la correlación
cols_corr = [
    'Evasion', 'Adulto_Mayor', 'Pareja', 'Dependientes', 'Meses_Contrato',
    'Servicio_Telefono', 'Factura_Electronica', 'Cargo_Mensual',
    'Cargo_Total', 'Cuentas_Diarias', 'Num_Servicios'
]
corr_matrix = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(cols_corr)))
ax.set_yticks(range(len(cols_corr)))
ax.set_xticklabels(cols_corr, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(cols_corr, fontsize=9)

for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color='black' if abs(val) < 0.6 else 'white')

ax.set_title('Matriz de Correlación — Variables de Clientes', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n🔍 Correlaciones con Evasion (ordenadas):')
print(corr_matrix['Evasion'].drop('Evasion').sort_values(ascending=False).round(3))

## 💾 Exportar datos limpios para Parte 2 (ML)

In [ ]:
# Guardar el DataFrame limpio como CSV para usarlo en la Parte 2
df.to_csv('TelecomX_tratado.csv', index=False)
print('✅ Archivo guardado: TelecomX_tratado.csv')
print(f'   Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head(3)

# 📄 Informe Final

---

## 🔹 Introducción

Telecom X enfrenta una alta tasa de cancelaciones que amenaza su rentabilidad. Este análisis aplica un pipeline **ETL completo** sobre los datos de 7,267 clientes para identificar los factores asociados a la evasión (churn), con el objetivo de proporcionar insumos al equipo de Data Science para el desarrollo de modelos predictivos y estrategias de retención.

---

## 🔹 Limpieza y Tratamiento de Datos

| Tarea | Acción realizada |
|---|---|
| Estructura JSON anidada | Normalización con `json_normalize` |
| Renombre de columnas | Traducción al español para mayor claridad |
| `Cargo_Total` como string | Conversión a `float64` con `pd.to_numeric` |
| Nulos en `Cargo_Total` | Imputación con mediana |
| Variables binarias (Yes/No) | Codificación a 1/0 |
| Nueva variable | `Cuentas_Diarias` = Cargo_Mensual / 30 |
| Nueva variable | `Num_Servicios` = suma de servicios activos |
| Duplicados | Verificados y eliminados |

---

## 🔹 Análisis Exploratorio — Hallazgos Clave

### 📊 Tasa de Churn Global
Aproximadamente el **26–27%** de los clientes cancelaron el servicio, lo que representa una tasa de evasión significativa que requiere atención inmediata.

### 📊 Variables con mayor impacto en el Churn

| Variable | Hallazgo |
|---|---|
| **Tipo de Contrato** | Clientes con contrato **mes a mes** tienen tasas de churn ~3x mayores que los de contratos anuales o bianuales |
| **Meses de Contrato** | Clientes nuevos (pocos meses) evaden mucho más; a mayor antigüedad, menor churn |
| **Servicio de Internet** | Clientes con **Fibra óptica** presentan churn más alto que los de DSL o sin internet |
| **Método de Pago** | El pago por **cheque electrónico** tiene la mayor tasa de churn |
| **Adulto Mayor** | Los adultos mayores presentan mayor tendencia a la evasión |
| **Cargo Mensual** | Clientes que evaden tienen cargos mensuales promedio más altos |
| **Num. Servicios** | No hay una relación lineal clara: ni pocos ni muchos servicios garantizan retención |
| **Género** | No hay diferencia significativa entre géneros |

---

## 🔹 Conclusiones e Insights

1. **El tipo de contrato es el predictor más fuerte**: Los contratos mes a mes son el mayor factor de riesgo. Incentivar contratos anuales reduciría significativamente el churn.
2. **La antigüedad protege contra la evasión**: Los primeros meses son críticos. Un programa de onboarding y retención temprana es prioritario.
3. **La Fibra óptica genera más insatisfacción**: A pesar de ser el servicio más premium, sus clientes evaden más. Posiblemente por precio elevado vs. expectativas.
4. **El pago por cheque electrónico es señal de alerta**: Podría estar correlacionado con clientes menos comprometidos o con dificultades de pago.
5. **El cargo mensual alto aumenta el riesgo**: Clientes con facturación alta sin servicios de valor agregado son vulnerables.

---

## 🔹 Recomendaciones Estratégicas

| Estrategia | Descripción |
|---|---|
| 🎯 **Incentivos para contratos largos** | Descuentos o beneficios exclusivos para clientes que migren a contratos anuales |
| 🚀 **Programa de bienvenida** | Acompañamiento intensivo en los primeros 3 meses (mayor riesgo de churn) |
| 💰 **Revisión de precios de Fibra** | Evaluar si el precio justifica la experiencia percibida; ofrecer beneficios adicionales |
| 📲 **Promover métodos de pago automático** | Migrar clientes de cheque electrónico a débito automático o tarjeta |
| 👴 **Atención diferenciada a adultos mayores** | Soporte técnico simplificado y planes especiales para este segmento |
| 📦 **Bundles de servicios** | Ofrecer paquetes combinados con descuento para aumentar el compromiso del cliente |
